In [ ]:
from unsloth import FastLanguageModel
from tqdm import tqdm
model_name = "marcelbinz/Llama-3.1-Centaur-70B-adapter"
model, tokenizer = FastLanguageModel.from_pretrained(
  model_name = model_name,
  max_seq_length = 32768,
  dtype = None,
  load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2024.10.7: Fast Llama patching. Transformers = 4.44.2.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.151 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.5.1+cu124. CUDA = 8.0. CUDA Toolkit = 12.4.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Unsloth: We fixed a gradient accumulation bug, but it seems like you don't have the latest transformers version!
Please update transformers, TRL and unsloth via:
`pip install --upgrade --no-cache-dir unsloth git+https://github.com/huggingface/transformers.git git+https://github.com/huggingface/trl.git`
Unsloth 2024.10.7 patched 80 layers with 80 QKV layers, 80 O layers and 80 MLP layers.


In [ ]:
org_game = """
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request? Tell me the number and the reason in the following example format, and nothing else:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"]= "false"
FastLanguageModel.for_inference(model)

input_text = org_game

inputs = tokenizer(input_text, return_tensors="pt", padding=True, return_attention_mask=True).to("cuda")

output_ids = model.generate(
    inputs.input_ids,
    attention_mask = inputs.attention_mask,#attention mask
    max_length=200, 
    #do_sample=False,
    temperature=0.5,
    )
# Decode
output_text = tokenizer.decode(output_ids[0],skip_special_tokens=True)
print(output_text)

In [ ]:
lst = []
for _ in tqdm(range(1000)):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, return_attention_mask=True).to("cuda")
    
    # 清除历史记忆，确保每次都是独立推理
    output_ids = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask, # attention mask
        max_length=200, 
        temperature=0.5
    )
    
    # 解码生成的文本并保存
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    lst.append(output_text)